# PSF Caching Benchmark on Real Rubin Data

This notebook benchmarks PSF caching on the same Rubin coadd injection path used in the main injection notebook.

What it measures:
- Wall time for the actual injection step
- Speedup from PSF caching
- Timing breakdown by injection stage
- Cache hit rate on the real coadd PSF

This notebook is meant to run on RSP or another environment with the Rubin stack and Butler access.

In [ ]:
import os
import sys
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.inject import inject_clusters_rubin_psf, PSFCache

from lsst.daf.butler import Butler
from lsst.geom import Point2D

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

os.makedirs('../plots', exist_ok=True)
print('Imports successful.')

## 1. Load the Actual Rubin Coadd and PSF

This uses the same DP0.2 coadd image and coadd PSF object as the main injection workflow.

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')
data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd = butler.get('deepCoadd', dataId=data_id)

CUTOUT = 1200
BASE_IMAGE = coadd.image.array[:CUTOUT, :CUTOUT].copy()
psf_obj = coadd.getPsf()
bbox = coadd.getBBox()
BBOX_X_MIN = bbox.getMinX()
BBOX_Y_MIN = bbox.getMinY()

print(f'Image shape : {BASE_IMAGE.shape}')
print(f'BBox offset : ({BBOX_X_MIN}, {BBOX_Y_MIN})')
print('PSF object  :', type(psf_obj).__name__)

## 2. Estimate a PSF Fallback FWHM

This samples the real coadd PSF across the cutout and uses the median determinant radius as the fallback FWHM value.

In [ ]:
N_GRID = 5
xs = np.linspace(BBOX_X_MIN + 100, bbox.getMaxX() - 100, N_GRID).astype(int)
ys = np.linspace(BBOX_Y_MIN + 100, bbox.getMaxY() - 100, N_GRID).astype(int)

grid_fwhm = np.full((N_GRID, N_GRID), np.nan)
for i, y in enumerate(ys):
    for j, x in enumerate(xs):
        try:
            shape = psf_obj.computeShape(Point2D(int(x), int(y)))
            grid_fwhm[i, j] = shape.getDeterminantRadius() * 2.355
        except Exception:
            pass

valid = grid_fwhm[~np.isnan(grid_fwhm)]
PSF_FWHM_FALLBACK = float(np.median(valid)) if len(valid) > 0 else 3.5

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(grid_fwhm, origin='lower', cmap='viridis')
plt.colorbar(im, ax=ax, label='FWHM (px)')
ax.set_title(f'PSF FWHM across coadd\nmedian={PSF_FWHM_FALLBACK:.3f} px')
plt.tight_layout()
plt.show()

print(f'PSF fallback FWHM: {PSF_FWHM_FALLBACK:.3f} px')

## 3. Build the Real Injection Catalog

This uses the same injection pipeline and catalog generator as the main notebook, but keeps the sample size small enough for a benchmark.

In [ ]:
BENCHMARK_N_CLUSTERS = 300
BENCHMARK_TRIALS = 3

config = InjectionConfig(
    run_name='psf_cache_benchmark',
    n_clusters=BENCHMARK_N_CLUSTERS,
    seed=42,
    edge_buffer=60,
    add_noise=True,
    use_actual_psf=True,
    save_injected_image=False,
    output_dir='outputs/psf_cache_benchmark',
    tract=data_id['tract'],
    patch=data_id['patch'],
    band=data_id['band'],
    pixel_scale=0.2,
    zero_point=27.0,
    cluster_config=ClusterConfig(
        profile_type='king',
        mag_min=20.0,
        mag_max=26.0,
        r_half_min=2.0,
        r_half_max=10.0,
        concentration_min=5.0,
        concentration_max=30.0,
        age_min_gyr=1.0,
        age_max_gyr=13.0,
    ),
)

pipe = InjectionPipeline(config)
pipe.load_data(image=BASE_IMAGE)
catalog = pipe.generate_catalog()

print(f'Catalog size: {len(catalog)} clusters')
pd.DataFrame(catalog).head()

## 4. Define the Benchmark Runner

Each trial uses the same real Rubin image and the same catalog. The only thing that changes is whether PSF caching is enabled.

In [ ]:
def run_real_data_benchmark(n_trials, use_cache):
    trial_results = []

    for trial in range(n_trials):
        trial_image = BASE_IMAGE.copy()
        psf_cache = PSFCache(max_entries=500, grid_size=8) if use_cache else None

        start = time.perf_counter()
        result = inject_clusters_rubin_psf(
            image=trial_image,
            catalog=catalog,
            psf_obj=psf_obj,
            bbox_x_min=BBOX_X_MIN,
            bbox_y_min=BBOX_Y_MIN,
            psf_fwhm_fallback=PSF_FWHM_FALLBACK,
            pixel_scale=config.pixel_scale,
            zero_point=config.zero_point,
            add_noise=config.add_noise,
            use_actual_psf=config.use_actual_psf,
            rng_seed=config.seed,
            verbose=False,
            use_psf_cache=use_cache,
            psf_cache=psf_cache,
        )
        elapsed = time.perf_counter() - start

        if len(result) == 4:
            injected_image, injection_info, timing, cache_stats = result
        else:
            injected_image, injection_info = result
            timing = {}
            cache_stats = None

        trial_results.append({
            'trial': trial + 1,
            'use_cache': use_cache,
            'wall_time': elapsed,
            'n_injected': len(injection_info),
            'timing': timing,
            'cache_stats': cache_stats,
            'success': True,
        })

    return trial_results

print('Benchmark runner ready.')

## 5. Run Baseline and Cached Trials

The baseline run uses the real Rubin PSF without caching. The cached run uses the same exact workload with PSF caching enabled.

In [ ]:
print(f'Running {BENCHMARK_TRIALS} baseline trials and {BENCHMARK_TRIALS} cached trials...')
baseline_results = run_real_data_benchmark(BENCHMARK_TRIALS, use_cache=False)
cached_results = run_real_data_benchmark(BENCHMARK_TRIALS, use_cache=True)
all_results = baseline_results + cached_results

print('Done.')
print(f'Baseline trials: {len(baseline_results)}')
print(f'Cached trials  : {len(cached_results)}')

## 6. Compute the Summary Statistics

This converts the raw trial results into the headline numbers for your advisor: average wall time, standard deviation, speedup, and cache hit rate.

In [ ]:
baseline_times = [r['wall_time'] for r in baseline_results if r['success']]
cached_times = [r['wall_time'] for r in cached_results if r['success']]

baseline_mean = float(np.mean(baseline_times))
baseline_std = float(np.std(baseline_times))
cached_mean = float(np.mean(cached_times))
cached_std = float(np.std(cached_times))
speedup = baseline_mean / cached_mean

baseline_timing_keys = ['profile_build', 'psf_fetch', 'convolution', 'placement']
baseline_timing_mean = {key: float(np.mean([r['timing'].get(key, 0.0) for r in baseline_results if r['success']])) for key in baseline_timing_keys}
cached_timing_mean = {key: float(np.mean([r['timing'].get(key, 0.0) for r in cached_results if r['success']])) for key in baseline_timing_keys}

cache_hit_rates = [r['cache_stats']['hit_rate_pct'] for r in cached_results if r['cache_stats'] is not None]
mean_cache_hit_rate = float(np.mean(cache_hit_rates)) if cache_hit_rates else float('nan')

print('=' * 72)
print('REAL-DATA BENCHMARK RESULTS')
print('=' * 72)
print(f'Baseline mean wall time: {baseline_mean:.3f} s')
print(f'Baseline std dev       : {baseline_std:.3f} s')
print(f'Cached mean wall time   : {cached_mean:.3f} s')
print(f'Cached std dev          : {cached_std:.3f} s')
print(f'Speedup                 : {speedup:.2f}x')
print(f'Time saved per run      : {baseline_mean - cached_mean:.3f} s')
print(f'Mean cache hit rate     : {mean_cache_hit_rate:.1f}%')
print('\nTiming stage means (seconds):')
for key in baseline_timing_keys:
    print(f'  {key:14s} baseline={baseline_timing_mean[key]:.3f}  cached={cached_timing_mean[key]:.3f}')
print('=' * 72)

## 7. Plot: Wall Time Comparison

This is the main advisor-facing plot. It compares total injection wall time with and without caching on the real Rubin image.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categories = ['Baseline\n(no cache)', 'With PSF\ncache']
means = [baseline_mean, cached_mean]
stds = [baseline_std, cached_std]
colors = ['#ff7f0e', '#2ca02c']

axes[0].bar(categories, means, yerr=stds, capsize=10, color=colors, alpha=0.75, edgecolor='black', linewidth=1.2)
axes[0].set_ylabel('Wall Time (seconds)')
axes[0].set_title(f'Actual Rubin Injection Speed Comparison\n({BENCHMARK_N_CLUSTERS} clusters, {BENCHMARK_TRIALS} trials)')
for i, mean in enumerate(means):
    axes[0].text(i, mean + stds[i] + 0.01, f'{mean:.3f}s', ha='center', fontweight='bold')

axes[1].barh(['Speedup'], [speedup], color='#1f77b4', alpha=0.75, edgecolor='black', linewidth=1.2)
axes[1].set_xlabel('Speedup (x)')
axes[1].set_title('Performance Improvement')
axes[1].text(speedup / 2, 0, f'{speedup:.2f}x', ha='center', va='center', fontweight='bold', color='white', fontsize=13)

plt.tight_layout()
plt.savefig('../plots/real_data_benchmark_speedup.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved plot to ../plots/real_data_benchmark_speedup.png')

## 8. Plot: Timing Breakdown by Stage

This plot shows where the time goes inside the injection step. The point of caching is to reduce the PSF fetch stage, while leaving the rest of the pipeline unchanged.

In [ ]:
stages = baseline_timing_keys
x = np.arange(len(stages))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width / 2, [baseline_timing_mean[s] for s in stages], width, label='Baseline', color='#ff7f0e', alpha=0.75, edgecolor='black', linewidth=1.0)
bars2 = ax.bar(x + width / 2, [cached_timing_mean[s] for s in stages], width, label='With PSF Cache', color='#2ca02c', alpha=0.75, edgecolor='black', linewidth=1.0)

ax.set_ylabel('Time (seconds)')
ax.set_title('Timing Breakdown by Injection Stage on Real Rubin Data')
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', ' ').title() for s in stages])
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bars in (bars1, bars2):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.3f}s', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../plots/real_data_timing_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved plot to ../plots/real_data_timing_breakdown.png')

## 9. Summary for Advisor

This final cell prints a short table and the main talking points you can use directly in a presentation.

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Average Wall Time',
        'Std Dev',
        'Speedup Factor',
        'Time Saved per Run',
        'PSF Cache Hit Rate',
        'Catalog Size',
    ],
    'Baseline': [
        f'{baseline_mean:.3f} s',
        f'{baseline_std:.3f} s',
        '1.0x',
        '—',
        'N/A',
        f'{len(catalog)}',
    ],
    'With PSF Cache': [
        f'{cached_mean:.3f} s',
        f'{cached_std:.3f} s',
        f'{speedup:.2f}x',
        f'{baseline_mean - cached_mean:.3f} s',
        f'{mean_cache_hit_rate:.1f}%',
        f'{len(catalog)}',
    ],
})

psf_fetch_share = cached_timing_mean['psf_fetch'] / sum(cached_timing_mean.values()) * 100 if sum(cached_timing_mean.values()) > 0 else 0.0

print('\n' + '=' * 80)
print('BENCHMARK SUMMARY FOR ADVISOR')
print('=' * 80 + '\n')
print(summary.to_string(index=False))
print('\n' + '=' * 80)
print('\nMain takeaway: the benchmark uses the real Rubin coadd image and the actual injection code path, so the measured speedup reflects the pipeline you are really running.')
print(f'PSF fetch takes {psf_fetch_share:.1f}% of the cached run time on average.')
print(f'Cache hit rate averaged {mean_cache_hit_rate:.1f}% across cached trials.')